In [1]:
import numpy as np

X = np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y = np.array([[0.],[1.],[1.],[0.]])

def sigmoid(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    exp_z = np.exp(z[~pos])
    out[~pos] = exp_z / (1.0 + exp_z)
    return out

def forward(X, params):
    z1 = X @ params["W1"] + params["b1"]
    a1 = np.tanh(z1)
    z2 = a1 @ params["W2"] + params["b2"]
    a2 = sigmoid(z2)
    return {"z1": z1, "a1": a1, "z2": z2, "a2": a2}

def bce_loss(y, y_hat, eps=1e-12):
    y = np.asarray(y, dtype=float)
    y_hat = np.asarray(y_hat, dtype=float)
    p = np.clip(y_hat, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1.0 - y) * np.log(1.0 - p)))

def backprop(X, y, params, cache):
    m = X.shape[0]
    a1 = cache["a1"]
    a2 = cache["a2"]

    dz2 = (a2 - y) / m
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    da1 = dz2 @ params["W2"].T
    dz1 = da1 * (1.0 - a1**2)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

def gradient_step(params, grads, lr):
    return {
        "W1": params["W1"] - lr * grads["dW1"],
        "b1": params["b1"] - lr * grads["db1"],
        "W2": params["W2"] - lr * grads["dW2"],
        "b2": params["b2"] - lr * grads["db2"],
    }


In [2]:
rng = np.random.default_rng(0)
params = {
    "W1": rng.normal(0,1,(2,8)),
    "b1": np.zeros((1,8)),
    "W2": rng.normal(0,1,(8,1)),
    "b2": np.zeros((1,1))
}

for _ in range(6000):
    cache = forward(X, params)
    grads = backprop(X, y, params, cache)
    params = gradient_step(params, grads, 0.5)

pred = (forward(X, params)["a2"] > 0.5).astype(int).ravel()
print(pred)
print(round(bce_loss(y, forward(X, params)["a2"]), 4))


[0 1 1 0]
0.0004
